# Arabica Coffee Quality Analysis — Mini Project
**Branch**: CSBS | **Theme**: Agriculture  
**Problem Statement**: Analyzing the effect of altitude, processing method, variety, and origin on Arabica coffee cup quality.

---
### Step 1 & 2: Data Cleaning & Preprocessing
This notebook imports modular cleaning functions from  and inspects the pipeline.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import coffee_utils as cu

RAW_DATA_PATH = os.path.join("data", "arabica_ratings_raw.csv")
CLEAN_DATA_PATH = os.path.join("data", "arabica_ratings_cleaned.csv")
print("coffee_utils imported successfully!")

### 1. Load Raw Data & Inspect Initial Shape

In [ ]:
df_raw = cu.load_raw(RAW_DATA_PATH)
print(f"Raw dataset shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head(3)

### 2. Drop Scraping Artifacts and Junk Columns

In [ ]:
df_dropped = cu.drop_junk_columns(df_raw)
print(f"Columns reduced from {df_raw.shape[1]} to {df_dropped.shape[1]}")

### 3. Clean Total Cup Points & Strip Text Prefixes

In [ ]:
df_points = cu.clean_cup_points(df_dropped)
print(df_points["total_cup_points"].describe())

### 4. Clean Complex Altitude Text Entries into Metric Meters (m)

In [ ]:
df_alt = cu.clean_altitude(df_points)
print(f"Parsed valid altitudes: {df_alt['altitude_m'].notna().sum()} / {len(df_alt)}")
print(df_alt["altitude_m"].describe())

### 5. Standardize Categorical Variables (Processing Method, Variety, Country)

In [ ]:
df_cat = cu.standardize_categories(df_alt, top_varieties=8, min_country_samples=10)
print("Processing Method Counts:
", df_cat["processing_method_clean"].value_counts(dropna=False))
print("
Variety Counts:
", df_cat["variety_clean"].value_counts(dropna=False))

### 6. Outlier Analysis & Filtering Corrupted Entries

In [ ]:
df_outliers, stats = cu.flag_outliers(df_cat, column="total_cup_points", drop_corrupted=True, min_valid_score=60.0)
print("IQR Statistics:", stats)

### 7. Run Full Pipeline via Orchestrator Function

In [ ]:
df_clean = cu.get_clean_dataset(RAW_DATA_PATH, categorical_strategy="unknown", save_clean_path=CLEAN_DATA_PATH)
print(f"Cleaned dataset saved! Shape: {df_clean.shape}")
df_clean[["total_cup_points", "altitude_m", "processing_method_clean", "variety_clean", "country_clean"]].head(10)

---
### Step 3: Exploratory Data Visualization
In this section, we generate publication-grade visualizations addressing each research question.

In [ ]:
OUTPUTS_DIR = "outputs"
plot_paths = cu.generate_all_visualizations(df_clean, OUTPUTS_DIR)
print("All 6 visualizations generated successfully in:", OUTPUTS_DIR)

#### Figure 1: Distribution of Total Cup Points (Histogram + KDE)
Demonstrates negative skewness and clustering around the SCA Specialty benchmark (80.0–85.0).

In [ ]:
from IPython.display import Image, display
display(Image(filename=plot_paths["cup_points_distribution"]))

#### Figure 2: Total Cup Points by Processing Method
Compares Washed, Natural, Semi-Washed, and Pulped Natural/Honey methods.

In [ ]:
display(Image(filename=plot_paths["processing_methods"]))

#### Figure 3: Total Cup Points by Variety
Compares major cultivars ordered by median cup points.

In [ ]:
display(Image(filename=plot_paths["varieties"]))

#### Figure 4: Average Cup Points by Country of Origin
Compares origins with at least 15 lots using 95% confidence intervals.

In [ ]:
display(Image(filename=plot_paths["countries"]))

#### Figure 5: Altitude vs. Total Cup Points (OLS Regression)
Directly addresses Research Question 1 on elevation impact.

In [ ]:
display(Image(filename=plot_paths["altitude_regression"]))

#### Figure 6: Correlation Heatmap Across Growing Conditions, Defects & Quality
Triangular correlation matrix of numeric variables.

In [ ]:
display(Image(filename=plot_paths["correlation_heatmap"]))

---
### Step 4: Statistical Analysis & Hypothesis Testing
We formally test normality and homoscedasticity to choose between parametric (ANOVA) and non-parametric (Kruskal-Wallis) tests, conduct post-hoc analyses, and fit Multiple Linear Regression.

#### 1. Processing Method Tests (Research Question 2)
- **Normality**: Shapiro-Wilk test per processing method
- **Homoscedasticity**: Levene's test for equal variances
- **Hypothesis Test**: Kruskal-Wallis H-test vs One-Way ANOVA
- **Post-Hoc**: Tukey HSD and Pairwise Mann-Whitney U with Holm correction

In [ ]:
norm_pm = cu.run_normality_and_variance_tests(df_clean, "processing_method_clean")
print("--- Processing Method Normality (Shapiro-Wilk) ---")
for grp, res in norm_pm["shapiro_by_group"].items():
    print(f"{grp:22s} (n={res['n']}): W={res['shapiro_w']:.4f}, p={res['p_value']:.4e} -> Normal: {res['is_normal"]}")
print(f"
Levene's Test: Stat={norm_pm['levene_stat']:.4f}, p={norm_pm['levene_p']:.4e} (Equal Variances: {norm_pm['equal_variances"]})")
print(f"Recommended Test: {norm_pm['recommended_test"]}")

tests_pm = cu.run_group_hypothesis_tests(df_clean, "processing_method_clean")
print(f"
One-Way ANOVA F: {tests_pm['anova_f']:.4f}, p={tests_pm['anova_p']:.4e}")
print(f"Kruskal-Wallis H: {tests_pm['kruskal_h']:.4f}, p={tests_pm['kruskal_p']:.4e} (Significant: {tests_pm['kruskal_significant"]})")

posthoc_pm = cu.run_posthoc_tests(df_clean, "processing_method_clean")
print("
--- Pairwise Mann-Whitney U (Holm-Adjusted) ---")
display(posthoc_pm["mann_whitney_summary"])

#### 2. Botanical Varieties Tests (Research Question 3)
- Normality (Shapiro-Wilk) and Homoscedasticity (Levene's)
- Kruskal-Wallis vs ANOVA
- Holm-adjusted Pairwise Mann-Whitney U post-hoc comparisons

In [ ]:
norm_var = cu.run_normality_and_variance_tests(df_clean, "variety_clean")
print(f"Levene's Test across Varieties: Stat={norm_var['levene_stat']:.4f}, p={norm_var['levene_p']:.4e}")
print(f"Recommended Test: {norm_var['recommended_test"]}")

tests_var = cu.run_group_hypothesis_tests(df_clean, "variety_clean")
print(f"
One-Way ANOVA F: {tests_var['anova_f']:.4f}, p={tests_var['anova_p']:.4e}")
print(f"Kruskal-Wallis H: {tests_var['kruskal_h']:.4f}, p={tests_var['kruskal_p']:.4e}")

posthoc_var = cu.run_posthoc_tests(df_clean, "variety_clean")
print("
Significant Pairwise Variety Differences (Holm-Adjusted):")
display(posthoc_var["mann_whitney_significant"])

#### 3. Multiple Linear Regression & Variance Decomposition (Research Question 4)
Model:   
- Evaluated via **Type II ANOVA** to determine which factor explains the most unique variance.
- Uses **HC3 Robust Standard Errors** to adjust for heteroscedasticity.

In [ ]:
reg_res = cu.fit_multiple_regression(df_clean)
print(f"Sample Size (n): {reg_res['n_obs"]:,}")
print(f"R-Squared: {reg_res['r_squared"]:.4f} | Adj R-Squared: {reg_res['adj_r_squared"]:.4f}")
print(f"Overall F-Stat: {reg_res['f_stat"]:.2f}, p={reg_res['f_pvalue']:.4e}")

print("
--- Type II ANOVA Table (Factor Variance Explained) ---")
display(reg_res["anova_table"][["sum_sq", "df", "F", "PR(>F)", "partial_eta_sq", "pct_model_variance"]])

print("
--- Significant Predictor Coefficients (HC3 Robust SE) ---")
display(reg_res["coefficients_significant"])

#### Figure 7: Multiple Linear Regression Diagnostic Plots
- Panel A: Residuals vs. Fitted Values (Linearity & Homoscedasticity)
- Panel B: Normal Q-Q Plot of Residuals

In [ ]:
diag_path = os.path.join("outputs", "07_regression_diagnostics.png")
display(Image(filename=diag_path))